# Location Geotagger — JMU Fallback Review

Interactive tool to visually inspect and batch-correct geoparsed locations.

**Workflow:**

1. Run the **Setup** cell (Cell 2)
2. Run the **Geotagger** cell (Cell 3) — the map and panel appear below it
3. Use the map to inspect dots, the geocoder to look up correct coordinates, and **Apply** to save corrections
4. Click **💾 Save** periodically to write progress back to `JMU_geoparsed_long.csv`
5. When done, run **Part D** in `lesson_4_4_preparing_review_sheet.ipynb` to export the cleaned file

**Batch tip:** Type a place name in the **Filter** box → click **Highlight** → all matching dots turn purple. Click one, geocode the correct location, check *Apply to ALL rows with same place name*, then Apply.


In [7]:
# Install dependencies if not present
import subprocess, sys

def _ensure(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'], check=True)
        __import__(import_name or pkg)

_ensure('geopy')
_ensure('anywidget')

import pandas as pd
import plotly.graph_objects as go
import ipywidgets as w
from IPython.display import display

try:
    from geopy.geocoders import Nominatim
    from geopy.extra.rate_limiter import RateLimiter
    _locator = Nominatim(user_agent='jmu_geotagger_v1')
    _geocode = RateLimiter(_locator.geocode, min_delay_seconds=1)
    GEO_OK = True
except Exception:
    GEO_OK = False
    print("⚠️  geopy unavailable — geocoder disabled.")

DATA_PATH   = '../data/JMU/JMU_geoparsed_long.csv'
PLACE_TYPES = ['Country', 'State', 'Region', 'City', 'Neighborhood',
               'University', 'Road', 'Building', 'Natural Feature', '']

df = pd.read_csv(DATA_PATH, dtype=str).fillna('')

print(f"✅  Loaded {len(df):,} rows  ·  {df['place'].nunique()} unique place names")
print(f"\nCurrent review progress:")
vc = df['action'].value_counts(dropna=False)
for k, v in vc.items():
    print(f"  {k if k else '(unreviewed)'}: {v:,}")


✅  Loaded 884 rows  ·  322 unique place names

Current review progress:
  (unreviewed): 884


In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────────────
COLOR = {'KEEP': '#2ca02c', 'CORRECT': '#ff7f0e', 'REMOVE': '#d62728', '': '#aec7e8'}
HIGHLIGHT = '#9467bd'

def build_summary(df):
    return (
        df.groupby(['place', 'latitude', 'longitude'], dropna=False, sort=False)
          .agg(count               = ('place',              'size'),
               place_type          = ('place_type',         'first'),
               action              = ('action',             'first'),
               corrected_name      = ('corrected_name',     'first'),
               corrected_latlon    = ('corrected_latlon',   'first'),
               corrected_place_type= ('corrected_place_type','first'))
          .reset_index()
    )

def make_trace(s, highlight=None):
    lat = pd.to_numeric(s['latitude'],  errors='coerce')
    lon = pd.to_numeric(s['longitude'], errors='coerce')
    ok  = lat.notna() & lon.notna()
    vs  = s[ok].copy()
    colors = [
        HIGHLIGHT if (highlight and r['place'] in highlight)
        else COLOR.get(r['action'], '#aec7e8')
        for _, r in vs.iterrows()
    ]
    hover = [
        '<b>' + r['place'] + '</b><br>'
        + 'type: '     + (r['place_type'] or '—') + '<br>'
        + 'mentions: ' + str(r['count'])           + '<br>'
        + 'action: '   + (r['action'] or 'unreviewed')
        for _, r in vs.iterrows()
    ]
    return go.Scattermap(
        lat=lat[ok].values, lon=lon[ok].values,
        mode='markers',
        marker=dict(size=10, color=colors, opacity=0.85),
        hovertext=hover, hoverinfo='text',
        text=vs['place'].values,
        customdata=vs.index.values,
    ), vs

def _split_latlon(s):
    """Parse 'lat, lon' string → (lat_str, lon_str). Returns ('', '') if invalid."""
    s = s.strip()
    if ',' in s:
        parts = s.split(',', 1)
        return parts[0].strip(), parts[1].strip()
    return s, ''

# ── Initial build ─────────────────────────────────────────────────────────────
summary    = build_summary(df)
trace0, _  = make_trace(summary)

fig = go.FigureWidget(
    data=[trace0],
    layout=go.Layout(
        map=dict(style='carto-positron', center=dict(lat=37.5, lon=-78.0), zoom=5),
        height=520,
        margin=dict(l=0, r=0, t=30, b=0),
        title_text='Click a dot to select it',
    )
)

# ── Widgets ───────────────────────────────────────────────────────────────────
w_info     = w.HTML('<i>Click a dot on the map to select it.</i>')
w_name     = w.Text(description='Name:',
                    placeholder='corrected name (blank = keep original)',
                    layout=w.Layout(width='400px'))
w_latlon   = w.Text(description='Coords:',
                    placeholder='paste Google Maps  e.g. 38.4339, -78.8729',
                    layout=w.Layout(width='420px'))
w_lat_in   = w.Text(description='Lat:',  placeholder='auto-filled from Coords or Geocode',
                    layout=w.Layout(width='240px'))
w_lon_in   = w.Text(description='Lon:',  placeholder='auto-filled from Coords or Geocode',
                    layout=w.Layout(width='240px'))
w_type     = w.Dropdown(description='Type:', options=PLACE_TYPES,
                        layout=w.Layout(width='280px'))
w_action   = w.ToggleButtons(options=['KEEP', 'CORRECT', 'REMOVE'],
                             style={'button_width': '90px'})
w_scope    = w.Checkbox(value=False,
                        description='Apply to ALL rows with same place name',
                        style={'description_width': 'initial'})
w_apply    = w.Button(description='✅  Apply', button_style='success',
                      layout=w.Layout(width='130px'))

w_query    = w.Text(placeholder='e.g. James Madison University Harrisonburg VA',
                    layout=w.Layout(width='420px'))
w_geo_btn  = w.Button(description='🔍 Geocode', button_style='info',
                      layout=w.Layout(width='120px'))
w_geo_out  = w.HTML()

w_filter     = w.Text(placeholder='Filter / highlight by name (e.g. JMU)',
                      layout=w.Layout(width='300px'))
w_filter_btn = w.Button(description='Highlight', layout=w.Layout(width='100px'))
w_filter_clr = w.Button(description='Clear',     layout=w.Layout(width='75px'))

w_save     = w.Button(description='💾  Save', button_style='warning',
                      layout=w.Layout(width='110px'))
w_save_out = w.Output()

# ── Mutable state ─────────────────────────────────────────────────────────────
_sel = [None]
_hi  = [None]

# ── Auto-split Google Maps paste into lat/lon display fields ──────────────────
def on_latlon_change(change):
    lat, lon = _split_latlon(change['new'])
    w_lat_in.value = lat
    w_lon_in.value = lon

w_latlon.observe(on_latlon_change, names='value')

# ── Map refresh ───────────────────────────────────────────────────────────────
def refresh():
    global summary
    summary = build_summary(df)
    t, _    = make_trace(summary, highlight=_hi[0])
    with fig.batch_update():
        fig.data[0].lat          = t.lat
        fig.data[0].lon          = t.lon
        fig.data[0].text         = t.text
        fig.data[0].hovertext    = t.hovertext
        fig.data[0].marker.color = t.marker.color
        fig.data[0].customdata   = t.customdata

# ── Click ──────────────────────────────────────────────────────────────────────
def on_click(trace, points, _state):
    if not points.point_inds:
        return
    sidx    = int(trace.customdata[points.point_inds[0]])
    _sel[0] = sidx
    row     = summary.loc[sidx]
    w_info.value = (
        '<b>' + row['place'] + '</b>'
        + '&nbsp;·&nbsp;type: <code>' + (row['place_type'] or '—') + '</code>'
        + '&nbsp;·&nbsp;mentions: <b>' + str(row['count']) + '</b><br>'
        + '<small>lat ' + row['latitude'] + ' · lon ' + row['longitude'] + '</small>'
    )
    w_name.value   = row['corrected_name'] or ''
    ll = row['corrected_latlon'] or ''
    w_latlon.value = ll
    lat, lon       = _split_latlon(ll)
    w_lat_in.value = lat
    w_lon_in.value = lon
    t = row['corrected_place_type'] or row['place_type'] or ''
    w_type.value   = t if t in PLACE_TYPES else ''
    a = row['action']
    w_action.value = a if a in ('KEEP', 'CORRECT', 'REMOVE') else 'KEEP'

fig.data[0].on_click(on_click)

# ── Geocode ────────────────────────────────────────────────────────────────────
def on_geocode(_btn):
    q = w_query.value.strip()
    if not q:
        return
    if not GEO_OK:
        w_geo_out.value = '<span style="color:red">geopy not available.</span>'
        return
    w_geo_out.value = '⏳ searching…'
    res = _geocode(q)
    if res:
        lat = str(round(res.latitude,  6))
        lon = str(round(res.longitude, 6))
        w_latlon.value  = lat + ', ' + lon   # also triggers auto-split observer
        w_geo_out.value = '<span style="color:green">✅ ' + res.address + '</span>'
    else:
        w_geo_out.value = '<span style="color:red">Not found — try a more specific query.</span>'

w_geo_btn.on_click(on_geocode)

# ── Apply ─────────────────────────────────────────────────────────────────────
def on_apply(_btn):
    sidx = _sel[0]
    if sidx is None:
        w_info.value = '<span style="color:orange">⚠️  Click a dot first.</span>'
        return
    row    = summary.loc[sidx]
    action = w_action.value
    if w_scope.value:
        mask = df['place'] == row['place']
    else:
        mask = ((df['place']     == row['place']) &
                (df['latitude']  == row['latitude']) &
                (df['longitude'] == row['longitude']))
    df.loc[mask, 'action']         = action
    df.loc[mask, 'corrected_name'] = w_name.value
    if action == 'CORRECT':
        df.loc[mask, 'corrected_latlon']      = w_latlon.value
        df.loc[mask, 'corrected_place_type']  = w_type.value
    else:
        df.loc[mask, 'corrected_latlon']      = ''
        df.loc[mask, 'corrected_place_type']  = ''
    refresh()
    n   = int(mask.sum())
    who = 'all "' + row['place'] + '"' if w_scope.value else 'this dot (' + row['place'] + ')'
    w_info.value = '<span style="color:green">✅ ' + action + ' → ' + str(n) + ' row(s) — ' + who + '</span>'

w_apply.on_click(on_apply)

# ── Filter / highlight ────────────────────────────────────────────────────────
def on_filter(_btn):
    q      = w_filter.value.strip().lower()
    _hi[0] = {p for p in summary['place'] if q in p.lower()} if q else None
    refresh()

def on_clear(_btn):
    _hi[0]         = None
    w_filter.value = ''
    refresh()

w_filter_btn.on_click(on_filter)
w_filter_clr.on_click(on_clear)

# ── Save ──────────────────────────────────────────────────────────────────────
def on_save(_btn):
    df.to_csv(DATA_PATH, index=False)
    rev = df['action'].isin(['KEEP', 'CORRECT', 'REMOVE']).sum()
    pct = rev / len(df) * 100 if len(df) else 0
    with w_save_out:
        w_save_out.clear_output()
        print(f"💾  Saved → {DATA_PATH}")
        print(f"   {rev:,} / {len(df):,} rows reviewed ({pct:.0f}%)")
        for k, v in df['action'].value_counts(dropna=False).items():
            print(f"   {k if k else '(unreviewed)'}: {v:,}")

w_save.on_click(on_save)

# ── Layout ────────────────────────────────────────────────────────────────────
legend = w.HTML(
    '<div style="font-size:12px;margin:4px 0">'
    '<span style="background:#aec7e8;padding:2px 8px;border-radius:3px">◉ unreviewed</span>&nbsp;'
    '<span style="background:#2ca02c;color:#fff;padding:2px 8px;border-radius:3px">◉ KEEP</span>&nbsp;'
    '<span style="background:#ff7f0e;color:#fff;padding:2px 8px;border-radius:3px">◉ CORRECT</span>&nbsp;'
    '<span style="background:#d62728;color:#fff;padding:2px 8px;border-radius:3px">◉ REMOVE</span>&nbsp;'
    '<span style="background:#9467bd;color:#fff;padding:2px 8px;border-radius:3px">◉ highlighted</span>'
    '</div>'
)

edit_panel = w.VBox([
    w.HTML('<b>Selected location</b>'),
    w_info,
    w.HTML('<hr style="margin:5px 0">'),
    w.HTML('<b>Action</b>'),
    w_action,
    w.HTML('<b>Corrections</b> <small>(leave blank to keep original value)</small>'),
    w_name,
    w.HTML('<small>Paste Google Maps coordinates <i>or</i> use the geocoder below:</small>'),
    w_latlon,
    w.HBox([w_lat_in, w_lon_in]),
    w_type,
    w.HTML('<hr style="margin:5px 0">'),
    w.HTML('<b>Geocoder</b> <small>— fills Coords above</small>'),
    w.HBox([w_query, w_geo_btn]),
    w_geo_out,
    w.HTML('<hr style="margin:5px 0">'),
    w_scope,
    w_apply,
], layout=w.Layout(padding='10px', border='1px solid #ccc',
                   min_width='480px', max_width='520px'))

display(w.VBox([
    w.HBox([w_filter, w_filter_btn, w_filter_clr]),
    legend,
    fig,
    w.HBox([edit_panel]),
    w.HBox([w_save, w_save_out]),
]))


ValueError: 
Invalid property path 'map._derived' for layout


ValueError: 
Invalid property path 'map._derived' for layout


ValueError: 
Invalid property path 'map._derived' for layout


ValueError: 
Invalid property path 'map._derived' for layout


ValueError: 
Invalid property path 'map._derived' for layout


...

ValueError: 
Invalid property path 'map._derived' for layout


ValueError: 
Invalid property path 'map._derived' for layout


ValueError: 
Invalid property path 'map._derived' for layout


ValueError: 
Invalid property path 'map._derived' for layout


ValueError: 
Invalid property path 'map._derived' for layout
